In [20]:
import pandas as pd
import numpy as np
from pathlib import Path

files = sorted(Path("../data/raw").glob("seriea_*.csv"))
df = pd.concat([pd.read_csv(f).assign(Season=f.stem[-4:]) for f in files], ignore_index=True)
df = df[["Season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]].dropna()
df[["FTHG", "FTAG"]] = df[["FTHG", "FTAG"]].astype(int)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed")
df = df.sort_values("Date").reset_index(drop=True)

C:\Users\Utente\AppData\Local\Temp\ipykernel_23000\631019114.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = pd.concat([pd.read_csv(f).assign(Season=f.stem[-4:]) for f in files], ignore_index=True)


In [21]:
K, HOME_ADV = 20, 25
ratings = {}
elo_h, elo_a = [], []
for row in df.itertuples():
    rh = ratings.get(row.HomeTeam, 1500)
    ra = ratings.get(row.AwayTeam, 1500)
    elo_h.append(rh)
    elo_a.append(ra)
    exp_home = 1 / (1 + 10 ** ((ra - (rh + HOME_ADV)) / 400))
    score = {"H": 1, "D": 0.5, "A": 0}[row.FTR]
    ratings[row.HomeTeam] = rh + K * (score - exp_home)
    ratings[row.AwayTeam] = ra - K * (score - exp_home)

df["elo_home"] = elo_h
df["elo_away"] = elo_a
df["elo_diff"] = df["elo_home"] - df["elo_away"]

In [22]:
casa = df[["Date", "HomeTeam", "FTHG", "FTAG", "FTR"]].rename(
    columns={"HomeTeam": "team", "FTHG": "gf", "FTAG": "gs"})
casa["punti"] = casa["FTR"].map({"H": 3, "D": 1, "A": 0})
casa["in_casa"] = 1

trasf = df[["Date", "AwayTeam", "FTAG", "FTHG", "FTR"]].rename(
    columns={"AwayTeam": "team", "FTAG": "gf", "FTHG": "gs"})
trasf["punti"] = trasf["FTR"].map({"A": 3, "D": 1, "H": 0})
trasf["in_casa"] = 0

lunga = pd.concat([casa, trasf]).drop(columns="FTR").sort_values(["team", "Date"])

In [23]:
g = lunga.groupby("team")
for n in [5, 20]:
    for col in ["punti", "gf", "gs"]:
        lunga[f"{col}_{n}"] = g[col].transform(
            lambda s: s.shift(1).rolling(n, min_periods=1).mean())

lunga["riposo"] = g["Date"].diff().dt.days.clip(upper=14)

In [24]:
lunga[lunga["team"] == "Inter"][["Date", "punti", "punti_5", "gf", "gf_5", "riposo"]].tail(7)

,Date,punti,punti_5,gf,gf_5,riposo
3843,2026-09-19,1,2.6,2,3.2,5.0
3851,2026-10-10,3,2.6,3,3.0,14.0
3862,2026-10-17,3,2.6,3,2.8,7.0
3875,2026-10-25,3,2.6,2,3.2,8.0
3887,2026-10-28,3,2.6,2,3.0,3.0
3892,2026-10-31,0,2.6,1,2.4,3.0
3906,2026-11-08,3,2.4,2,2.2,8.0


In [25]:
feat = ["punti_5", "gf_5", "gs_5", "punti_20", "gf_20", "gs_20", "riposo"]
h = lunga[lunga["in_casa"] == 1][feat].add_prefix("home_")
a = lunga[lunga["in_casa"] == 0][feat].add_prefix("away_")
tab = df.join(h).join(a)

tab.to_csv("../data/processed/feature.csv", index=False)
print(tab.shape)
tab.isna().sum()[lambda s: s > 0]

(3910, 24)


home_punti_5     14
home_gf_5        14
home_gs_5        14
home_punti_20    14
home_gf_20       14
home_gs_20       14
home_riposo      14
away_punti_5     20
away_gf_5        20
away_gs_5        20
away_punti_20    20
away_gf_20       20
away_gs_20       20
away_riposo      20
dtype: int64

In [26]:
tab.groupby("FTR")[["elo_diff", "home_punti_5", "away_punti_5", "home_gf_20", "away_gf_20"]].mean().round(2)

,elo_diff,home_punti_5,away_punti_5,home_gf_20,away_gf_20
FTR,,,,,
A,-80.66,1.18,1.61,1.22,1.56
D,-5.00,1.29,1.38,1.32,1.36
H,63.99,1.53,1.24,1.51,1.25
